In [1]:
import torch
from torch import nn, optim
import torchvision
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import random_split, DataLoader
import numpy as np
from math import isqrt

In [2]:
from model import get_pos_embeddings, Net

In [3]:
# IMage
D_image = 96
N_CHANNELS = 3
PATCH_SIZE = 8
D_patch = (PATCH_SIZE**2) * N_CHANNELS # 192
N_PATCHES = (D_image**2) // (PATCH_SIZE**2) # 144
N_ROWS = D_image // PATCH_SIZE

# Encoder
D = 384
N_HEADS = 3
Dk = D // N_HEADS # 64
D_mlp = 4*D

# Decoder
# Normally D_decoder = D//2 but ViT-Tiny is already so shallow, and I can reuse
# positional embeddings for encoder/decoder if they're the same dim
D_decoder = D
D_decoder_mlp = 4*D_decoder

BATCH_SIZE = 256

PERCENT_UNMASKED = 0.25

In [4]:
# Transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4467, 0.4398, 0.4066), (0.2603, 0.2566, 0.2713))
])

In [5]:
# Download and load datasets
supervised_trainset = torchvision.datasets.STL10(
    root='./data', split='train', download=True, transform=transform
)
testset = torchvision.datasets.STL10(
    root='./data', split='test', download=True, transform=transform
)
unlabeled_set = torchvision.datasets.STL10(
    root='./data', split='unlabeled', download=True, transform=transform
)
# DataLoaders
testloader = DataLoader(
    testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)
ssl_trainloader = DataLoader(
    unlabeled_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2
)

val_size = 500 # 10%
train_size = len(supervised_trainset) - val_size
generator = torch.Generator().manual_seed(12) # reproducible split
train_subset, val_subset = random_split(
    supervised_trainset, [train_size, val_size], generator=generator
)
supervised_trainloader = DataLoader(
    train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2
)
supervised_valloader = DataLoader(
    val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

In [6]:
device = torch.device('cuda:0')

In [7]:
pos_embeddings = get_pos_embeddings(D, N_PATCHES, N_ROWS, device)

In [8]:
net = Net(
    n_encoder_blocks=8,
    n_decoder_blocks=2,
    D_image=D_image,
    patch_size=PATCH_SIZE,
    D_patch=D_patch,
    n_patches=N_PATCHES,
    n_rows=N_ROWS,
    D=D,
    D_mlp=D_mlp,
    D_decoder=D_decoder,
    D_decoder_mlp=D_decoder_mlp,
    n_heads=N_HEADS,
    pos_embeddings_enc=pos_embeddings,
    pos_embedding_dec=pos_embeddings,
    percent_unmasked=PERCENT_UNMASKED)

In [10]:
# net.load_state_dict(torch.load('mae_pretrain_no_aug_0.259_loss.pth'))
# net.load_state_dict(torch.load('mae_pretrain_aug_0.208_192.pth'))
net.load_state_dict(torch.load('mae_pretrain_8e_2d.pth'))
net.to(device)

Net(
  (img2enc_projection): Linear(in_features=192, out_features=384, bias=True)
  (encoder_blocks): ModuleList(
    (0-7): 8 x ModuleDict(
      (norm_a): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (msa): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=384, out_features=384, bias=True)
      )
      (norm_b): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp_a): Linear(in_features=384, out_features=1536, bias=True)
      (mlp_b): Linear(in_features=1536, out_features=384, bias=True)
    )
  )
  (enc2dec_projection): Linear(in_features=384, out_features=384, bias=True)
  (decoder_blocks): ModuleList(
    (0-1): 2 x ModuleDict(
      (norm_a): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (msa): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=384, out_features=384, bias=True)
      )
      (norm_b): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp_a): Li

In [14]:
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR, ConstantLR

net.eval()
N_CLASSES = 10
N_EPOCHS = 50

WARMUP_EPOCHS = 5
HOLD_EPOCHS = 10
COSINE_EPOCHS = N_EPOCHS - WARMUP_EPOCHS - HOLD_EPOCHS # 25

linear_probe = nn.Linear(D_decoder, N_CLASSES).to(device)
optimizer_probe = optim.AdamW(linear_probe.parameters(), lr=1e-3, weight_decay=0.0)

"""
The point of warmup is to avoid blowing up early in training, 
when the probe's weights are random and gradients can be large or unstable. 
AdamW especially benefits because its adaptive moments (m, v) need a few steps to stabilize
 — starting at full LR before they've stabilized can push weights in a bad direction.
"""
warmup = LinearLR(optimizer_probe, start_factor=0.01, total_iters=WARMUP_EPOCHS)
cosine = CosineAnnealingLR(optimizer_probe, T_max=N_EPOCHS - WARMUP_EPOCHS)
hold = ConstantLR(optimizer_probe, factor=1.0, total_iters=HOLD_EPOCHS)
scheduler = SequentialLR(optimizer_probe, [warmup, hold, cosine], milestones=[WARMUP_EPOCHS, WARMUP_EPOCHS + HOLD_EPOCHS])

criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
best_state = None
best_epoch = -1

for epoch in range(N_EPOCHS):
    # --- train ---
    linear_probe.train()
    running_loss = 0.0
    n_batches = 0
    for i, data in enumerate(supervised_trainloader):
        inputs, labels = data[0].to(device), data[1].to(device)
        with torch.no_grad():
            embeddings, _, _, _ = net.encode(inputs)
            reps = embeddings.mean(dim=1)
        logits = linear_probe(reps)
        loss = criterion(logits, labels)

        optimizer_probe.zero_grad()
        loss.backward()
        optimizer_probe.step()

        running_loss += loss.item()
        n_batches += 1

        if i % 10 == 9:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / (i + 1):.6f}')

    scheduler.step()
    train_loss = running_loss / n_batches

    # --- validate ---
    linear_probe.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in supervised_valloader:
            inputs, labels = data[0].to(device), data[1].to(device)
            embeddings, _, _, _ = net.encode(inputs)
            reps = embeddings.mean(dim=1)
            logits = linear_probe(reps)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total

    current_lr = optimizer_probe.param_groups[0]['lr']
    print(f'epoch {epoch + 1:3d} | train_loss={train_loss:.4f} | '
          f'val_acc={val_acc:.4f} | lr={current_lr:.2e}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_state = {k: v.detach().clone() for k, v in linear_probe.state_dict().items()}

# load best-validation weights back into the probe
linear_probe.load_state_dict(best_state)
linear_probe.eval()

print(f'Finished training. Best val_acc={best_val_acc:.4f} at epoch {best_epoch}')

[1,    10] loss: 2.402458
epoch   1 | train_loss=2.3983 | val_acc=0.0900 | lr=2.08e-04
[2,    10] loss: 2.351744
epoch   2 | train_loss=2.3146 | val_acc=0.2220 | lr=4.06e-04
[3,    10] loss: 2.159506
epoch   3 | train_loss=2.1101 | val_acc=0.3720 | lr=6.04e-04
[4,    10] loss: 1.926110
epoch   4 | train_loss=1.8675 | val_acc=0.5140 | lr=8.02e-04
[5,    10] loss: 1.679468
epoch   5 | train_loss=1.6327 | val_acc=0.5620 | lr=1.00e-03
[6,    10] loss: 1.466036
epoch   6 | train_loss=1.4476 | val_acc=0.5920 | lr=1.00e-03
[7,    10] loss: 1.341583
epoch   7 | train_loss=1.3220 | val_acc=0.6180 | lr=1.00e-03
[8,    10] loss: 1.259202
epoch   8 | train_loss=1.2337 | val_acc=0.6300 | lr=1.00e-03
[9,    10] loss: 1.196032
epoch   9 | train_loss=1.1763 | val_acc=0.6340 | lr=1.00e-03
[10,    10] loss: 1.135047
epoch  10 | train_loss=1.1274 | val_acc=0.6340 | lr=1.00e-03
[11,    10] loss: 1.106728
epoch  11 | train_loss=1.0952 | val_acc=0.6700 | lr=1.00e-03
[12,    10] loss: 1.070435
epoch  12 | tr

In [15]:
# Now how do I test it
linear_probe.eval()
correct = 0
total = 0

with torch.no_grad():
    for i, data in enumerate(testloader):
        inputs, labels = data[0].to(device), data[1].to(device)
        embeddings, _, _, _ = net.encode(inputs)
        reps = embeddings.mean(dim=1)
        logits = linear_probe(reps)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.6787
